# 1. Ingestion
### Load dữ liệu từ file CSV local lên

In [1]:
from src.modules.ingestion.ingestion_execute import ingestion_execution

ingestion_execution()

2026-07-03 16:16:55 | INFO | src.modules.ingestion.ingestion_execute | Starting Ingestion Execution...
2026-07-03 16:16:55 | INFO | src.modules.ingestion.ingestion_execute | Config loaded successfully.
2026-07-03 16:16:55 | INFO | src.modules.ingestion.ingestion_execute | Starting Spark job...
2026-07-03 16:16:56 | INFO | src.modules.ingestion.ingestion_execute | :: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
2026-07-03 16:16:56 | INFO | src.modules.ingestion.ingestion_execute | Ivy Default Cache set to: /root/.ivy2/cache
2026-07-03 16:16:56 | INFO | src.modules.ingestion.ingestion_execute | The jars for the packages stored in: /root/.ivy2/jars
2026-07-03 16:16:56 | INFO | src.modules.ingestion.ingestion_execute | org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
2026-07-03 16:16:56 | INFO | src.modules.ingestion.ingestion_execute | org.apache.iceberg#iceberg-aws-bundle added as a dependency
2026-0

# 2. AI Classification
### - Load dữ liệu bảng từ MinIO
### - Tiến hành Regex Scan, LLM Scan
### - Ghi kết quả vào Metadata Store (Postgres)
#### Lưu ý: Nếu không truyền tên bảng thì Module sẽ quét toàn bộ các bảng trong

In [1]:
class TableNames:
    CITIZEN_INFO                = "citizen_info"
    ADMINISTRATIVE_RECORDS      = "administrative_records"
    HR_EMPLOYEES                = "hr_employees"
    TRANSACTION_LOGS            = "transaction_logs"
    FULL_TABLES                 = ""

TABLE_NAME = TableNames.CITIZEN_INFO

In [12]:
%load_ext autoreload
%autoreload 2

from src.modules.ai_governance.ai_governance_main import ai_governance_main

ai_governance_main(target_table=TableNames.CITIZEN_INFO)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
2026-07-03 17:23:30 | INFO | regex_output | ----------------------------------------
2026-07-03 17:23:30 | INFO | regex_output | Regex scan for column record_id:
2026-07-03 17:23:30 | INFO | regex_output | Regex status: UNDETERMINED
2026-07-03 17:23:30 | INFO | regex_output | All detected tags: []
2026-07-03 17:23:30 | INFO | regex_output | Regex candidates: []
2026-07-03 17:23:30 | INFO | regex_output | Raw scores: {}
2026-07-03 17:23:30 | INFO | regex_output | ----------------------------------------
2026-07-03 17:23:30 | INFO | regex_output | ----------------------------------------
2026-07-03 17:23:30 | INFO | regex_output | Regex scan for column cccd_so:
2026-07-03 17:23:30 | INFO | regex_output | Regex status: SUCCESS
2026-07-03 17:23:30 | INFO | regex_output | All detected tags: [(<SensitivityTag.RESIDENT_ID: 'RESIDENT_ID'>, 1.0)]
2026-07-03 17:23:30 | INFO | regex_output | Regex candidates: 

# 3. Policy Engine
### Query dữ liệu với Role cụ thể

#### 1. Role Admin

In [10]:
%load_ext autoreload
%autoreload 2

import logging
logging.getLogger().setLevel(logging.WARNING)

from src.core.dtos.enums import UserRole
from src.modules.policy_engine.policy_engine_main import policy_engine_main

df = policy_engine_main(table_name=TABLE_NAME, user_role=UserRole.ADMIN)[0]
df.limit(10).toPandas()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
2026-07-03 17:20:37 | INFO | policy_engine | Starting Policy Engine Pipeline...
2026-07-03 17:20:37 | INFO | policy_engine | Loading config...
2026-07-03 17:20:37 | INFO | policy_engine | Config loaded successfully.
2026-07-03 17:20:37 | INFO | policy_engine | Audit log written for user role ADMIN on table citizen_info


,record_id,cccd_so,ho_va_ten,ngay_sinh,gioi_tinh,sdt_chinh,dia_chi_cu_tru,quoc_tich,ref_ext,so_phu,nhom_mau,tinh_trang_hn,trinh_do_hv,nghe_nghiep,noi_sinh,so_nguoi_phu_thuoc,acc_status,dan_toc,e_type,xac_thuc
0,CIT-246316,027193234053,Đỗ Thị An,1993-04-06,Nam,0378078673,"112 Trường Chinh, Phường Bến Nghé, Quận 1, TP....",Việt Nam,do.thi.an575@yahoo.com,0924698379,AB,Ly hôn,Trung học phổ thông,Bác sĩ,"80 Bà Triệu, Phường Thu Thủy, TX. Cửa Lò, Nghệ An",2,Đã kích hoạt,Kinh,Tạm trú,VNeID định danh mức 2
1,CIT-581741,026087765179,Huỳnh Xuân Nam,1987-01-30,Nam,0902322047,"296 Bà Triệu, Xã Hàm Ninh, TP. Phú Quốc, Kiên ...",Việt Nam,huynh.xuan.nam47@yahoo.com,0832338687,B,Độc thân,Thạc sĩ,Lập trình viên,"84 Võ Thị Sáu, Xã Hương Vinh, TX. Hương Trà, T...",2,Đã kích hoạt,Nùng,Tạm trú,VNeID định danh mức 2
2,CIT-584714,074183171339,Bùi Thanh Hà,1983-12-17,Nữ,0824684531,"421 Lê Lợi, Phường Lam Sơn, TP. Hưng Yên, Hưng...",Việt Nam,bui.thanh.ha411@hotmail.com,0364539704,O,Đã kết hôn,Thạc sĩ,Kinh doanh tự do,"72 Trường Chinh, Phường Thuận Hòa, TP. Huế, Th...",5,Bị khóa,Khmer,Tạm trú,Chữ ký số công cộng
3,CIT-617488,024176534277,Lý Hữu Nga,1976-03-27,Nam,0352839607,"306 Trần Phú, Phường An Bình, Quận Ninh Kiều, ...",Việt Nam,ly.huu.nga391@protonmail.com,0969877065,O,Độc thân,Trung học phổ thông,Nông dân,"58 Hoàng Diệu, Xã Tam Giang, Huyện Yên Phong, ...",3,Đã kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
4,CIT-755674,092082912804,Đặng Phương Hà,1982-10-09,Nữ,0763564217,"307 Phan Đình Phùng, Phường Đông Thành, TP. Ni...",Việt Nam,dang.phuong.ha20@gmail.com,0896159230,B,Độc thân,Cao đẳng,Giáo viên,"418 Trần Phú, Phường Hòa Hiệp Nam, Quận Liên C...",4,Đã kích hoạt,Tày,Thông báo lưu trú,Chữ ký số công cộng
5,CIT-665492,087383222086,Đặng Đức Linh,1983-09-02,Nam,0847693979,"62 Trường Chinh, Phường Tây Sơn, TP. Tam Điệp,...",Việt Nam,dang.duc.linh66@hotmail.com,0334860684,B,Độc thân,Trung học phổ thông,Kỹ sư,"441 Phan Đình Phùng, Phường Hoàng Văn Thụ, TP....",0,Bị khóa,Tày,Tạm trú,Chữ ký số công cộng
6,CIT-922733,093392254801,Phạm Bảo Giang,1992-06-12,Nữ,0924194548,"221 Võ Thị Sáu, Phường Cầu Tre, Quận Ngô Quyền...",Việt Nam,pham.bao.giang421@outlook.com,0352651177,A,Góa,Đại học,Giáo viên,"275 Điện Biên Phủ, Phường Quang Trung, TP. Thá...",1,Chưa kích hoạt,Tày,Tạm trú,Chữ ký số công cộng
7,CIT-783823,082079053045,Phạm Xuân Châu,1979-11-17,Nam,0374965789,"247 Bà Triệu, Phường Lái Thiêu, Huyện Thuận An...",Việt Nam,pham.xuan.chau925@gmail.com,0597358113,A,Góa,Đại học,Công nhân,"80 Bà Triệu, Phường Văn An, TP. Chí Linh, Hải ...",2,Đã kích hoạt,Kinh,Thông báo lưu trú,VNeID định danh mức 2
8,CIT-994141,058362527276,Hồ Quốc Bình,1962-04-01,Nam,0359519948,"305 Trần Phú, Phường Hải Châu 1, Quận Hải Châu...",Việt Nam,ho.quoc.binh414@gmail.com,0791666754,A,Góa,Tiến sĩ,Nông dân,"343 Phan Đình Phùng, Phường Dị Sử, TX. Mỹ Hào,...",1,Chưa kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
9,CIT-663750,045060076819,Vũ Quốc Châu,1960-06-01,Nam,0813222252,"190 Hoàng Diệu, Phường Nam Bình, TP. Ninh Bình...",Việt Nam,vu.quoc.chau449@protonmail.com,0849874152,A,Ly hôn,Trung học phổ thông,Bác sĩ,"381 Phạm Văn Đồng, Phường Vũ Ninh, TP. Bắc Nin...",1,Chưa kích hoạt,Thái,Thông báo lưu trú,SMS OTP


#### 2. Role Analyst

In [13]:
%load_ext autoreload
%autoreload 2

import logging
logging.getLogger().setLevel(logging.WARNING)

from src.core.dtos.enums import UserRole
from src.modules.policy_engine.policy_engine_main import policy_engine_main

df = policy_engine_main(table_name=TABLE_NAME, user_role=UserRole.ANALYST)[0]
df.limit(10).toPandas()

2026-07-03 16:23:49 | INFO | policy_engine | Starting Policy Engine Pipeline...
2026-07-03 16:23:49 | INFO | policy_engine | Loading config...
2026-07-03 16:23:49 | INFO | policy_engine | Config loaded successfully.
2026-07-03 16:23:49 | INFO | src.modules.policy_engine.policy_engine_main | Connecting to infrastructure...
2026-07-03 16:23:49 | INFO | src.modules.policy_engine.policy_engine_main | Infrastructure connected successfully.
2026-07-03 16:23:49 | INFO | src.modules.policy_engine.policy_engine_main | Initializing Policy Engine Pipeline...
2026-07-03 16:23:49 | INFO | src.modules.policy_engine.policy_engine_main | Policy Engine Pipeline initialized successfully.
2026-07-03 16:23:49 | INFO | src.modules.policy_engine.policy_engine_main | Starting Policy Engine Pipeline...
2026-07-03 16:23:49 | INFO | src.modules.policy_engine.pipeline | Processing table: citizen_info
2026-07-03 16:23:49 | INFO | src.core.postgres.postgres_client | Connecting to Postgres ...
2026-07-03 16:23:49 |

,record_id,cccd_so,ho_va_ten,ngay_sinh,gioi_tinh,sdt_chinh,dia_chi_cu_tru,quoc_tich,ref_ext,so_phu,nhom_mau,tinh_trang_hn,trinh_do_hv,nghe_nghiep,noi_sinh,so_nguoi_phu_thuoc,acc_status,dan_toc,e_type,xac_thuc
0,CIT-246316,33f6d8f68541b1e02ed00891009cf226e04e9a8c31311b...,None,f6483138fe05a45a5628246eee5d53ce34653cfae12c39...,Nam,None,169daecc4ff56d1bac42f10f10e40f7ecbd8f8ff1f82b3...,Việt Nam,None,None,AB,Ly hôn,Trung học phổ thông,Bác sĩ,20149e0519a65da0f78af8846f417bfde74017d7bc6a2f...,2,Đã kích hoạt,Kinh,Tạm trú,VNeID định danh mức 2
1,CIT-581741,f928cd39e5df6c7faa0e776f4f67296b0a729b9ea3348e...,None,fd5b28b851a3fc89f5884f030432207b5174ca76a3c077...,Nam,None,9a0da3e76f22d1cce58336d0c09c03fdf2d3b62932b3ee...,Việt Nam,None,None,B,Độc thân,Thạc sĩ,Lập trình viên,d0a0d0ba60d8a072eb87425b78dc96995a815b2130f7d9...,2,Đã kích hoạt,Nùng,Tạm trú,VNeID định danh mức 2
2,CIT-584714,fa6758403ce7f77455053c729ee037ca1f32961088b5b5...,None,3f78ed2c3d413b3d9dd26bb8c3362ee8b8fcf1e7972fc3...,Nữ,None,e43831cf2358938332a47dfd31f00ea7eb65599fb54d06...,Việt Nam,None,None,O,Đã kết hôn,Thạc sĩ,Kinh doanh tự do,83acdd2c17e52167eac9ce5cdecc318a74081219baee88...,5,Bị khóa,Khmer,Tạm trú,Chữ ký số công cộng
3,CIT-617488,c1d8d4ff26a137335523b6467aab67c840503ffcb67113...,None,ced0526a5e0b0d1a552698e9acca8731d0cdc571d6ffba...,Nam,None,b2a857b3f4965ad18d06926898749738cb5aa58abe5890...,Việt Nam,None,None,O,Độc thân,Trung học phổ thông,Nông dân,c3dfdf3d5d8f286fa628bc62ab761130a32530031c4b0b...,3,Đã kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
4,CIT-755674,770775cf6de68f44a08ae2962a49ac6fbfa1e26b55052a...,None,1118196bc9d86d3366769677ab43ae7be09a6dcaabbcab...,Nữ,None,e27cc13a775784c8286c79b9aee3d3209cf482f3aa2f1b...,Việt Nam,None,None,B,Độc thân,Cao đẳng,Giáo viên,0d0ec181ba2e88bf0081d7e7c9f209ba40cbb0f202c59b...,4,Đã kích hoạt,Tày,Thông báo lưu trú,Chữ ký số công cộng
5,CIT-665492,ceeb5c52f0bbd0607810643149c434635a7e745e27dc41...,None,56ecc5ab6cc1d596bed61441aaf75bf76e364ca71f944e...,Nam,None,49119e22bb4e2e61a0a7cf5c329d2b5d59b7cef5770c16...,Việt Nam,None,None,B,Độc thân,Trung học phổ thông,Kỹ sư,a5f9a61242b5adcf9b60cdf50451e551d4b3ff3c222b61...,0,Bị khóa,Tày,Tạm trú,Chữ ký số công cộng
6,CIT-922733,291a1431d16ff1b58c784e23def102f2d0eef9e07a024c...,None,85af639a4cddcbe626d5ca62f4846f508b2fac27ac4a63...,Nữ,None,48717f296006f85c28ca5fd4257bfdec0c099b30dd16a1...,Việt Nam,None,None,A,Góa,Đại học,Giáo viên,5a6b79aa1287ebb90ed6f47b5f084cdc13e7bb57e30718...,1,Chưa kích hoạt,Tày,Tạm trú,Chữ ký số công cộng
7,CIT-783823,dc1fb38c5ccb15bb7b40a7d55b4e63194ae5aa5d360738...,None,dabe965d717ae4b4ee9f02c61884f7746eb62a826c146a...,Nam,None,ac7ec337dcbae5667c95485bfb1dd42b5714183b601c54...,Việt Nam,None,None,A,Góa,Đại học,Công nhân,75a097d3e51222c0838033bf97f9a0553d74c57f5f1eb4...,2,Đã kích hoạt,Kinh,Thông báo lưu trú,VNeID định danh mức 2
8,CIT-994141,21c017fd1837f5c6791ed80bbd37d11b07e9af52495f53...,None,12bd34182667c8926693a42a5d4a67a2946c718ace5ce1...,Nam,None,3dfe9f1a6b04617aac2d6daaef25dbce5e417e1dd176a5...,Việt Nam,None,None,A,Góa,Tiến sĩ,Nông dân,9a4492a666071eb0f8f4c8403aa829aa2a413260d66fd1...,1,Chưa kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
9,CIT-663750,69c008572019ab3c948f8cf9cfd89fb8934e9f12f22849...,None,1e89d631f0b45f95fd9770018b594cfd03d10b3dd6030d...,Nam,None,9bdfc8b86b307594fd6c5e83535a6de6e8d6ff979cec5f...,Việt Nam,None,None,A,Ly hôn,Trung học phổ thông,Bác sĩ,4d7006f1a2a3e3965b8b38555bda49139b65ad0f6712fc...,1,Chưa kích hoạt,Thái,Thông báo lưu trú,SMS OTP


#### 3. Role Auditor

In [6]:
%load_ext autoreload
%autoreload 2

import logging
logging.getLogger().setLevel(logging.WARNING)

from src.core.dtos.enums import UserRole
from src.modules.policy_engine.policy_engine_main import policy_engine_main

df = policy_engine_main(table_name=TABLE_NAME, user_role=UserRole.AUDITOR)[0]
df.limit(10).toPandas()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
2026-07-03 17:18:26 | INFO | policy_engine | Starting Policy Engine Pipeline...
2026-07-03 17:18:26 | INFO | policy_engine | Loading config...
2026-07-03 17:18:26 | INFO | policy_engine | Config loaded successfully.
2026-07-03 17:18:26 | INFO | policy_engine | Audit log written for user role AUDITOR on table citizen_info


,record_id,cccd_so,ho_va_ten,ngay_sinh,gioi_tinh,sdt_chinh,dia_chi_cu_tru,quoc_tich,ref_ext,so_phu,nhom_mau,tinh_trang_hn,trinh_do_hv,nghe_nghiep,noi_sinh,so_nguoi_phu_thuoc,acc_status,dan_toc,e_type,xac_thuc
0,CIT-246316,027*****4053,Đ* T** A*,REDACTED,Nam,037***8673,REDACTED,Việt Nam,d***********@yahoo.com,092***8379,AB,Ly hôn,Trung học phổ thông,Bác sĩ,REDACTED,2,Đã kích hoạt,Kinh,Tạm trú,VNeID định danh mức 2
1,CIT-581741,026*****5179,H**** X*** N**,REDACTED,Nam,090***2047,REDACTED,Việt Nam,h***************@yahoo.com,083***8687,B,Độc thân,Thạc sĩ,Lập trình viên,REDACTED,2,Đã kích hoạt,Nùng,Tạm trú,VNeID định danh mức 2
2,CIT-584714,074*****1339,B** T**** H*,REDACTED,Nữ,082***4531,REDACTED,Việt Nam,b**************@hotmail.com,036***9704,O,Đã kết hôn,Thạc sĩ,Kinh doanh tự do,REDACTED,5,Bị khóa,Khmer,Tạm trú,Chữ ký số công cộng
3,CIT-617488,024*****4277,L* H** N**,REDACTED,Nam,035***9607,REDACTED,Việt Nam,l************@protonmail.com,096***7065,O,Độc thân,Trung học phổ thông,Nông dân,REDACTED,3,Đã kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
4,CIT-755674,092*****2804,Đ*** P***** H*,REDACTED,Nữ,076***4217,REDACTED,Việt Nam,d***************@gmail.com,089***9230,B,Độc thân,Cao đẳng,Giáo viên,REDACTED,4,Đã kích hoạt,Tày,Thông báo lưu trú,Chữ ký số công cộng
5,CIT-665492,087*****2086,Đ*** Đ** L***,REDACTED,Nam,084***3979,REDACTED,Việt Nam,d**************@hotmail.com,033***0684,B,Độc thân,Trung học phổ thông,Kỹ sư,REDACTED,0,Bị khóa,Tày,Tạm trú,Chữ ký số công cộng
6,CIT-922733,093*****4801,P*** B** G****,REDACTED,Nữ,092***4548,REDACTED,Việt Nam,p****************@outlook.com,035***1177,A,Góa,Đại học,Giáo viên,REDACTED,1,Chưa kích hoạt,Tày,Tạm trú,Chữ ký số công cộng
7,CIT-783823,082*****3045,P*** X*** C***,REDACTED,Nam,037***5789,REDACTED,Việt Nam,p****************@gmail.com,059***8113,A,Góa,Đại học,Công nhân,REDACTED,2,Đã kích hoạt,Kinh,Thông báo lưu trú,VNeID định danh mức 2
8,CIT-994141,058*****7276,H* Q*** B***,REDACTED,Nam,035***9948,REDACTED,Việt Nam,h**************@gmail.com,079***6754,A,Góa,Tiến sĩ,Nông dân,REDACTED,1,Chưa kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
9,CIT-663750,045*****6819,V* Q*** C***,REDACTED,Nam,081***2252,REDACTED,Việt Nam,v**************@protonmail.com,084***4152,A,Ly hôn,Trung học phổ thông,Bác sĩ,REDACTED,1,Chưa kích hoạt,Thái,Thông báo lưu trú,SMS OTP


In [9]:
%load_ext autoreload
%autoreload 2

import logging
logging.getLogger().setLevel(logging.WARNING)

from IPython.display import display
from src.core.dtos.enums import UserRole
from src.modules.policy_engine.policy_engine_main import policy_engine_main

roles_to_check = [UserRole.ADMIN, UserRole.ANALYST, UserRole.AUDITOR]

for role in roles_to_check:
    print(f"{'='*25} Role: {role.name} {'='*25}")

    df = policy_engine_main(table_name=TABLE_NAME, user_role=role)[0]
    display(df.limit(10).toPandas())

    print("\n\n")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
========================= Role: ADMIN =========================
2026-07-03 17:19:06 | INFO | policy_engine | Starting Policy Engine Pipeline...
2026-07-03 17:19:06 | INFO | policy_engine | Loading config...
2026-07-03 17:19:06 | INFO | policy_engine | Config loaded successfully.
2026-07-03 17:19:06 | INFO | policy_engine | Audit log written for user role ADMIN on table citizen_info


,record_id,cccd_so,ho_va_ten,ngay_sinh,gioi_tinh,sdt_chinh,dia_chi_cu_tru,quoc_tich,ref_ext,so_phu,nhom_mau,tinh_trang_hn,trinh_do_hv,nghe_nghiep,noi_sinh,so_nguoi_phu_thuoc,acc_status,dan_toc,e_type,xac_thuc
0,CIT-246316,027193234053,Đỗ Thị An,1993-04-06,Nam,0378078673,"112 Trường Chinh, Phường Bến Nghé, Quận 1, TP....",Việt Nam,do.thi.an575@yahoo.com,0924698379,AB,Ly hôn,Trung học phổ thông,Bác sĩ,"80 Bà Triệu, Phường Thu Thủy, TX. Cửa Lò, Nghệ An",2,Đã kích hoạt,Kinh,Tạm trú,VNeID định danh mức 2
1,CIT-581741,026087765179,Huỳnh Xuân Nam,1987-01-30,Nam,0902322047,"296 Bà Triệu, Xã Hàm Ninh, TP. Phú Quốc, Kiên ...",Việt Nam,huynh.xuan.nam47@yahoo.com,0832338687,B,Độc thân,Thạc sĩ,Lập trình viên,"84 Võ Thị Sáu, Xã Hương Vinh, TX. Hương Trà, T...",2,Đã kích hoạt,Nùng,Tạm trú,VNeID định danh mức 2
2,CIT-584714,074183171339,Bùi Thanh Hà,1983-12-17,Nữ,0824684531,"421 Lê Lợi, Phường Lam Sơn, TP. Hưng Yên, Hưng...",Việt Nam,bui.thanh.ha411@hotmail.com,0364539704,O,Đã kết hôn,Thạc sĩ,Kinh doanh tự do,"72 Trường Chinh, Phường Thuận Hòa, TP. Huế, Th...",5,Bị khóa,Khmer,Tạm trú,Chữ ký số công cộng
3,CIT-617488,024176534277,Lý Hữu Nga,1976-03-27,Nam,0352839607,"306 Trần Phú, Phường An Bình, Quận Ninh Kiều, ...",Việt Nam,ly.huu.nga391@protonmail.com,0969877065,O,Độc thân,Trung học phổ thông,Nông dân,"58 Hoàng Diệu, Xã Tam Giang, Huyện Yên Phong, ...",3,Đã kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
4,CIT-755674,092082912804,Đặng Phương Hà,1982-10-09,Nữ,0763564217,"307 Phan Đình Phùng, Phường Đông Thành, TP. Ni...",Việt Nam,dang.phuong.ha20@gmail.com,0896159230,B,Độc thân,Cao đẳng,Giáo viên,"418 Trần Phú, Phường Hòa Hiệp Nam, Quận Liên C...",4,Đã kích hoạt,Tày,Thông báo lưu trú,Chữ ký số công cộng
5,CIT-665492,087383222086,Đặng Đức Linh,1983-09-02,Nam,0847693979,"62 Trường Chinh, Phường Tây Sơn, TP. Tam Điệp,...",Việt Nam,dang.duc.linh66@hotmail.com,0334860684,B,Độc thân,Trung học phổ thông,Kỹ sư,"441 Phan Đình Phùng, Phường Hoàng Văn Thụ, TP....",0,Bị khóa,Tày,Tạm trú,Chữ ký số công cộng
6,CIT-922733,093392254801,Phạm Bảo Giang,1992-06-12,Nữ,0924194548,"221 Võ Thị Sáu, Phường Cầu Tre, Quận Ngô Quyền...",Việt Nam,pham.bao.giang421@outlook.com,0352651177,A,Góa,Đại học,Giáo viên,"275 Điện Biên Phủ, Phường Quang Trung, TP. Thá...",1,Chưa kích hoạt,Tày,Tạm trú,Chữ ký số công cộng
7,CIT-783823,082079053045,Phạm Xuân Châu,1979-11-17,Nam,0374965789,"247 Bà Triệu, Phường Lái Thiêu, Huyện Thuận An...",Việt Nam,pham.xuan.chau925@gmail.com,0597358113,A,Góa,Đại học,Công nhân,"80 Bà Triệu, Phường Văn An, TP. Chí Linh, Hải ...",2,Đã kích hoạt,Kinh,Thông báo lưu trú,VNeID định danh mức 2
8,CIT-994141,058362527276,Hồ Quốc Bình,1962-04-01,Nam,0359519948,"305 Trần Phú, Phường Hải Châu 1, Quận Hải Châu...",Việt Nam,ho.quoc.binh414@gmail.com,0791666754,A,Góa,Tiến sĩ,Nông dân,"343 Phan Đình Phùng, Phường Dị Sử, TX. Mỹ Hào,...",1,Chưa kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
9,CIT-663750,045060076819,Vũ Quốc Châu,1960-06-01,Nam,0813222252,"190 Hoàng Diệu, Phường Nam Bình, TP. Ninh Bình...",Việt Nam,vu.quoc.chau449@protonmail.com,0849874152,A,Ly hôn,Trung học phổ thông,Bác sĩ,"381 Phạm Văn Đồng, Phường Vũ Ninh, TP. Bắc Nin...",1,Chưa kích hoạt,Thái,Thông báo lưu trú,SMS OTP





========================= Role: ANALYST =========================
2026-07-03 17:19:06 | INFO | policy_engine | Starting Policy Engine Pipeline...
2026-07-03 17:19:06 | INFO | policy_engine | Loading config...
2026-07-03 17:19:06 | INFO | policy_engine | Config loaded successfully.
2026-07-03 17:19:06 | INFO | policy_engine | Audit log written for user role ANALYST on table citizen_info


,record_id,cccd_so,ho_va_ten,ngay_sinh,gioi_tinh,sdt_chinh,dia_chi_cu_tru,quoc_tich,ref_ext,so_phu,nhom_mau,tinh_trang_hn,trinh_do_hv,nghe_nghiep,noi_sinh,so_nguoi_phu_thuoc,acc_status,dan_toc,e_type,xac_thuc
0,CIT-246316,33f6d8f68541b1e02ed00891009cf226e04e9a8c31311b...,None,f6483138fe05a45a5628246eee5d53ce34653cfae12c39...,Nam,None,169daecc4ff56d1bac42f10f10e40f7ecbd8f8ff1f82b3...,Việt Nam,None,None,AB,Ly hôn,Trung học phổ thông,Bác sĩ,20149e0519a65da0f78af8846f417bfde74017d7bc6a2f...,2,Đã kích hoạt,Kinh,Tạm trú,VNeID định danh mức 2
1,CIT-581741,f928cd39e5df6c7faa0e776f4f67296b0a729b9ea3348e...,None,fd5b28b851a3fc89f5884f030432207b5174ca76a3c077...,Nam,None,9a0da3e76f22d1cce58336d0c09c03fdf2d3b62932b3ee...,Việt Nam,None,None,B,Độc thân,Thạc sĩ,Lập trình viên,d0a0d0ba60d8a072eb87425b78dc96995a815b2130f7d9...,2,Đã kích hoạt,Nùng,Tạm trú,VNeID định danh mức 2
2,CIT-584714,fa6758403ce7f77455053c729ee037ca1f32961088b5b5...,None,3f78ed2c3d413b3d9dd26bb8c3362ee8b8fcf1e7972fc3...,Nữ,None,e43831cf2358938332a47dfd31f00ea7eb65599fb54d06...,Việt Nam,None,None,O,Đã kết hôn,Thạc sĩ,Kinh doanh tự do,83acdd2c17e52167eac9ce5cdecc318a74081219baee88...,5,Bị khóa,Khmer,Tạm trú,Chữ ký số công cộng
3,CIT-617488,c1d8d4ff26a137335523b6467aab67c840503ffcb67113...,None,ced0526a5e0b0d1a552698e9acca8731d0cdc571d6ffba...,Nam,None,b2a857b3f4965ad18d06926898749738cb5aa58abe5890...,Việt Nam,None,None,O,Độc thân,Trung học phổ thông,Nông dân,c3dfdf3d5d8f286fa628bc62ab761130a32530031c4b0b...,3,Đã kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
4,CIT-755674,770775cf6de68f44a08ae2962a49ac6fbfa1e26b55052a...,None,1118196bc9d86d3366769677ab43ae7be09a6dcaabbcab...,Nữ,None,e27cc13a775784c8286c79b9aee3d3209cf482f3aa2f1b...,Việt Nam,None,None,B,Độc thân,Cao đẳng,Giáo viên,0d0ec181ba2e88bf0081d7e7c9f209ba40cbb0f202c59b...,4,Đã kích hoạt,Tày,Thông báo lưu trú,Chữ ký số công cộng
5,CIT-665492,ceeb5c52f0bbd0607810643149c434635a7e745e27dc41...,None,56ecc5ab6cc1d596bed61441aaf75bf76e364ca71f944e...,Nam,None,49119e22bb4e2e61a0a7cf5c329d2b5d59b7cef5770c16...,Việt Nam,None,None,B,Độc thân,Trung học phổ thông,Kỹ sư,a5f9a61242b5adcf9b60cdf50451e551d4b3ff3c222b61...,0,Bị khóa,Tày,Tạm trú,Chữ ký số công cộng
6,CIT-922733,291a1431d16ff1b58c784e23def102f2d0eef9e07a024c...,None,85af639a4cddcbe626d5ca62f4846f508b2fac27ac4a63...,Nữ,None,48717f296006f85c28ca5fd4257bfdec0c099b30dd16a1...,Việt Nam,None,None,A,Góa,Đại học,Giáo viên,5a6b79aa1287ebb90ed6f47b5f084cdc13e7bb57e30718...,1,Chưa kích hoạt,Tày,Tạm trú,Chữ ký số công cộng
7,CIT-783823,dc1fb38c5ccb15bb7b40a7d55b4e63194ae5aa5d360738...,None,dabe965d717ae4b4ee9f02c61884f7746eb62a826c146a...,Nam,None,ac7ec337dcbae5667c95485bfb1dd42b5714183b601c54...,Việt Nam,None,None,A,Góa,Đại học,Công nhân,75a097d3e51222c0838033bf97f9a0553d74c57f5f1eb4...,2,Đã kích hoạt,Kinh,Thông báo lưu trú,VNeID định danh mức 2
8,CIT-994141,21c017fd1837f5c6791ed80bbd37d11b07e9af52495f53...,None,12bd34182667c8926693a42a5d4a67a2946c718ace5ce1...,Nam,None,3dfe9f1a6b04617aac2d6daaef25dbce5e417e1dd176a5...,Việt Nam,None,None,A,Góa,Tiến sĩ,Nông dân,9a4492a666071eb0f8f4c8403aa829aa2a413260d66fd1...,1,Chưa kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
9,CIT-663750,69c008572019ab3c948f8cf9cfd89fb8934e9f12f22849...,None,1e89d631f0b45f95fd9770018b594cfd03d10b3dd6030d...,Nam,None,9bdfc8b86b307594fd6c5e83535a6de6e8d6ff979cec5f...,Việt Nam,None,None,A,Ly hôn,Trung học phổ thông,Bác sĩ,4d7006f1a2a3e3965b8b38555bda49139b65ad0f6712fc...,1,Chưa kích hoạt,Thái,Thông báo lưu trú,SMS OTP





========================= Role: AUDITOR =========================
2026-07-03 17:19:06 | INFO | policy_engine | Starting Policy Engine Pipeline...
2026-07-03 17:19:06 | INFO | policy_engine | Loading config...
2026-07-03 17:19:06 | INFO | policy_engine | Config loaded successfully.
2026-07-03 17:19:06 | INFO | policy_engine | Audit log written for user role AUDITOR on table citizen_info


,record_id,cccd_so,ho_va_ten,ngay_sinh,gioi_tinh,sdt_chinh,dia_chi_cu_tru,quoc_tich,ref_ext,so_phu,nhom_mau,tinh_trang_hn,trinh_do_hv,nghe_nghiep,noi_sinh,so_nguoi_phu_thuoc,acc_status,dan_toc,e_type,xac_thuc
0,CIT-246316,027*****4053,Đ* T** A*,REDACTED,Nam,037***8673,REDACTED,Việt Nam,d***********@yahoo.com,092***8379,AB,Ly hôn,Trung học phổ thông,Bác sĩ,REDACTED,2,Đã kích hoạt,Kinh,Tạm trú,VNeID định danh mức 2
1,CIT-581741,026*****5179,H**** X*** N**,REDACTED,Nam,090***2047,REDACTED,Việt Nam,h***************@yahoo.com,083***8687,B,Độc thân,Thạc sĩ,Lập trình viên,REDACTED,2,Đã kích hoạt,Nùng,Tạm trú,VNeID định danh mức 2
2,CIT-584714,074*****1339,B** T**** H*,REDACTED,Nữ,082***4531,REDACTED,Việt Nam,b**************@hotmail.com,036***9704,O,Đã kết hôn,Thạc sĩ,Kinh doanh tự do,REDACTED,5,Bị khóa,Khmer,Tạm trú,Chữ ký số công cộng
3,CIT-617488,024*****4277,L* H** N**,REDACTED,Nam,035***9607,REDACTED,Việt Nam,l************@protonmail.com,096***7065,O,Độc thân,Trung học phổ thông,Nông dân,REDACTED,3,Đã kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
4,CIT-755674,092*****2804,Đ*** P***** H*,REDACTED,Nữ,076***4217,REDACTED,Việt Nam,d***************@gmail.com,089***9230,B,Độc thân,Cao đẳng,Giáo viên,REDACTED,4,Đã kích hoạt,Tày,Thông báo lưu trú,Chữ ký số công cộng
5,CIT-665492,087*****2086,Đ*** Đ** L***,REDACTED,Nam,084***3979,REDACTED,Việt Nam,d**************@hotmail.com,033***0684,B,Độc thân,Trung học phổ thông,Kỹ sư,REDACTED,0,Bị khóa,Tày,Tạm trú,Chữ ký số công cộng
6,CIT-922733,093*****4801,P*** B** G****,REDACTED,Nữ,092***4548,REDACTED,Việt Nam,p****************@outlook.com,035***1177,A,Góa,Đại học,Giáo viên,REDACTED,1,Chưa kích hoạt,Tày,Tạm trú,Chữ ký số công cộng
7,CIT-783823,082*****3045,P*** X*** C***,REDACTED,Nam,037***5789,REDACTED,Việt Nam,p****************@gmail.com,059***8113,A,Góa,Đại học,Công nhân,REDACTED,2,Đã kích hoạt,Kinh,Thông báo lưu trú,VNeID định danh mức 2
8,CIT-994141,058*****7276,H* Q*** B***,REDACTED,Nam,035***9948,REDACTED,Việt Nam,h**************@gmail.com,079***6754,A,Góa,Tiến sĩ,Nông dân,REDACTED,1,Chưa kích hoạt,Mường,Thường trú,Thẻ CCCD gắn chíp
9,CIT-663750,045*****6819,V* Q*** C***,REDACTED,Nam,081***2252,REDACTED,Việt Nam,v**************@protonmail.com,084***4152,A,Ly hôn,Trung học phổ thông,Bác sĩ,REDACTED,1,Chưa kích hoạt,Thái,Thông báo lưu trú,SMS OTP
